# Phase 4 - Modelling and comparison

Dans ce notebook, je compare les approches demandÃ©es dans le pipeline : baselines naÃ¯ves, puis un modÃ¨le principal de boosting (LightGBM/XGBoost) avec validation walk-forward.

Je respecte un test final sanctuarisÃ© (jours 57-62), non utilisÃ© pour ajuster les paramÃ¨tres.


## Plan d'Ã©valuation (walk-forward)

Je suis exactement le schÃ©ma du pipeline :
- Fold 1 : train jours 1-35, validation jours 36-42
- Fold 2 : train jours 1-42, validation jours 43-49
- Fold 3 : train jours 1-49, validation jours 50-56
- Test final sanctuarisÃ© : train jours 1-56, test jours 57-62

Les mÃ©triques calculÃ©es : MAE, RMSE et skill score vs meilleure baseline.


In [1]:
# Imports
from pathlib import Path
import warnings
import gc
import math
import copy
import numpy as np
import pandas as pd
import polars as pl

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")


In [ ]:
# Optional imports for models (graceful fallback)
# Uncomment in notebook runtime if Prophet is missing:
# %pip install prophet

HAVE_LGBM = False
HAVE_XGB = False
HAVE_PROPHET = False

try:
    import lightgbm as lgb
    HAVE_LGBM = True
except Exception:
    pass

try:
    import xgboost as xgb
    HAVE_XGB = True
except Exception:
    pass

try:
    from prophet import Prophet
    HAVE_PROPHET = True
except Exception:
    pass

print("LightGBM available:", HAVE_LGBM)
print("XGBoost available:", HAVE_XGB)
print("Prophet available:", HAVE_PROPHET)


In [3]:
# Paths and folders
project_root = Path("..")
input_path = project_root / "data" / "processed" / "features_target_600cells.parquet"
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

if not input_path.exists():
    raise FileNotFoundError(f"Missing features file: {input_path}")

print("Input:", input_path)
print("Reports dir:", reports_dir)


Input: c:\Users\hp\OneDrive\Desktop\projectTimeSeries\data\processed\features_target_600cells.parquet
Reports dir: c:\Users\hp\OneDrive\Desktop\projectTimeSeries\reports


In [4]:
# Load parquet
# We keep Polars for speed and convert to pandas when required by model APIs.
df = pl.read_parquet(input_path).sort(["square_id", "slot_30m"])

print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("Unique cells:", df["square_id"].n_unique())
df.head()


Shape: (1316362, 31)
Columns: 31
Unique cells: 499


square_id,slot_30m,internet_volume,sms_in,call_in,hour_slot,dow,sin_hour,cos_hour,sin_dow,cos_dow,is_weekend,lag_1,lag_2,lag_6,lag_12,lag_24,lag_48,lag_96,lag_336,roll_mean_3h,roll_std_3h,roll_max_3h,roll_mean_6h,roll_std_6h,roll_max_6h,roll_mean_24h,roll_std_24h,roll_max_24h,neighbor_mean_t_minus_2,target_1h
i32,i64,f64,f64,f64,i64,i64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3756,1383865200,1306.934109,24.730153,12.043053,46,0,-0.258819,0.965926,0.0,1.0,0,1127.112743,1179.635528,1242.46862,1014.797496,776.75967,1054.255931,1000.380946,778.354969,1251.32222,106.693811,1415.009148,1156.079633,129.67853,1415.009148,868.223051,221.565805,1415.009148,1344.487493,1141.257046
3756,1383867000,1108.578012,16.502316,4.402276,47,0,-0.130526,0.991445,0.0,1.0,0,1306.934109,1127.112743,1206.142168,1042.375933,789.788333,1047.03337,888.21607,765.975272,1262.066468,108.848068,1415.009148,1180.424351,128.156928,1415.009148,873.48718,228.958099,1415.009148,1247.145565,1029.775877
3756,1383868800,1141.257046,7.083825,2.629166,0,1,0.0,1.0,0.781831,0.62349,0,1108.578012,1306.934109,1337.565114,1093.34562,787.787603,1001.64614,802.755617,798.033506,1245.805776,124.967378,1415.009148,1185.941191,122.995005,1415.009148,874.76936,230.120025,1415.009148,1228.10927,790.6688
3756,1383870600,1029.775877,5.199155,1.501319,1,1,0.130526,0.991445,0.781831,0.62349,0,1141.257046,1108.578012,1415.009148,1064.012119,717.409338,901.746603,796.099546,1002.554008,1213.087764,121.796616,1415.009148,1189.933809,120.467621,1415.009148,877.677921,232.626425,1415.009148,1074.510722,778.299742
3756,1383872400,790.6688,7.961658,0.953884,2,1,0.258819,0.965926,0.781831,0.62349,0,1029.775877,1141.257046,1179.635528,987.874137,675.338164,710.040462,922.177936,826.738239,1148.882219,91.944515,1306.934109,1187.080789,124.072391,1415.009148,880.345197,233.640045,1415.009148,1063.872849,732.802194


## Construction d'un index temporel en jours

Le split est dÃ©fini en jours (1 Ã  62). Je reconstruis `day_idx` Ã  partir de `slot_30m`.


In [5]:
# Add day index aligned to original raw timeline (days 1..62)
raw_path = project_root / 'data' / 'processed' / 'work_600cells.parquet'
if not raw_path.exists():
    raise FileNotFoundError(f'Missing raw timeline reference file: {raw_path}')
raw_min_slot = pl.read_parquet(raw_path).select(pl.col('slot_30m').min()).item()

# 86400 seconds = 1 day, +1 => day number on original timeline
df = df.with_columns(
    (((pl.col('slot_30m') - raw_min_slot) // 86400) + 1).cast(pl.Int32).alias('day_idx')
)

print('Day range in features file:', df.select(pl.col('day_idx').min()).item(), '->', df.select(pl.col('day_idx').max()).item())


Day range in features file: 8 -> 62


In [6]:
# Split definitions from pipeline
folds = [
    {"fold": "fold_1", "train_days": (1, 35), "eval_days": (36, 42), "eval_type": "val"},
    {"fold": "fold_2", "train_days": (1, 42), "eval_days": (43, 49), "eval_type": "val"},
    {"fold": "fold_3", "train_days": (1, 49), "eval_days": (50, 56), "eval_type": "val"},
]

final_split = {"fold": "final_test", "train_days": (1, 56), "eval_days": (57, 62), "eval_type": "test"}

print("Folds configured:", [f["fold"] for f in folds] + [final_split["fold"]])


Folds configured: ['fold_1', 'fold_2', 'fold_3', 'final_test']


## Fonctions utilitaires

Je centralise les mÃ©triques, baselines et helper de split pour Ã©viter les erreurs de duplication.


In [7]:
def filter_day_range(data: pl.DataFrame, d0: int, d1: int) -> pl.DataFrame:
    return data.filter((pl.col('day_idx') >= d0) & (pl.col('day_idx') <= d1))


def mape_ignore_small(y_true: np.ndarray, y_pred: np.ndarray, min_true: float = 1.0) -> float:
    mask = y_true >= min_true
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / y_true[mask]) * 100.0)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    den = np.abs(y_true) + np.abs(y_pred)
    mask = den > 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(2.0 * np.abs(y_pred[mask] - y_true[mask]) / den[mask]) * 100.0)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = mape_ignore_small(y_true, y_pred, min_true=1.0)
    smape_v = smape(y_true, y_pred)
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan
    return {
        'MAE': float(mae),
        'RMSE': rmse,
        'MAPE': mape,
        'sMAPE': smape_v,
        'R2': r2,
    }


def quantile_mae(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    tmp = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
    tmp['traffic_quantile'] = pd.qcut(tmp['y_true'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
    out = {}
    for q, g in tmp.groupby('traffic_quantile', observed=False):
        out[f'MAE_{q}'] = float(mean_absolute_error(g['y_true'], g['y_pred'])) if len(g) > 0 else np.nan
    return out


def add_baseline_predictions(data: pl.DataFrame) -> pl.DataFrame:
    return data.with_columns([
        pl.col('internet_volume').alias('pred_persistence_simple'),
        pl.col('internet_volume').shift(46).over('square_id').alias('pred_persistence_seasonal'),
        pl.col('roll_mean_24h').alias('pred_moving_avg_24h'),
    ])


def evaluate_baselines(eval_df: pl.DataFrame, split_name: str) -> tuple[pd.DataFrame, pl.DataFrame]:
    dfb = add_baseline_predictions(eval_df)
    pred_cols = ['pred_persistence_simple', 'pred_persistence_seasonal', 'pred_moving_avg_24h']

    metrics_rows = []

    for col in pred_cols:
        temp = dfb.filter(pl.col(col).is_not_null())
        if temp.height == 0:
            metrics_rows.append({'split': split_name, 'model': col.replace('pred_', 'baseline_'), 'n_samples': 0})
            continue

        yt = temp['target_1h'].to_numpy()
        yp = temp[col].to_numpy()
        m = compute_metrics(yt, yp)
        q = quantile_mae(yt, yp)
        metrics_rows.append({'split': split_name, 'model': col.replace('pred_', 'baseline_'), 'n_samples': int(len(yt)), **m, **q})

    return pd.DataFrame(metrics_rows), dfb


## Ã‰valuation des baselines sur les 3 folds + test final


In [8]:
baseline_metrics_all = []
baseline_preds_all = []

for f in folds + [final_split]:
    eval_df = filter_day_range(df, f["eval_days"][0], f["eval_days"][1])
    metrics_df, pred_df = evaluate_baselines(eval_df, f["fold"])
    baseline_metrics_all.append(metrics_df)

    baseline_preds_all.append(
        pred_df.select([
            pl.lit(f["fold"]).alias("split"),
            "square_id", "slot_30m", "day_idx", "target_1h",
            "pred_persistence_simple", "pred_persistence_seasonal", "pred_moving_avg_24h"
        ])
    )

baseline_metrics = pd.concat(baseline_metrics_all, ignore_index=True)
baseline_predictions = pl.concat(baseline_preds_all)

baseline_metrics


,split,model,n_samples,MAE,RMSE,MAPE,sMAPE,R2,MAE_Q1,MAE_Q2,MAE_Q3,MAE_Q4
0,fold_1,baseline_persistence_simple,167664,173.803258,288.507110,13.609345,13.538959,0.915553,74.686553,133.675084,169.642193,317.209204
1,fold_1,baseline_persistence_seasonal,144710,240.847636,472.725966,19.745699,18.276616,0.774755,123.662127,172.590686,221.895197,445.240121
2,fold_1,baseline_moving_avg_24h,167664,474.632060,746.867076,47.005471,34.484850,0.434075,450.997713,344.844089,317.997700,784.688738
3,fold_2,baseline_persistence_simple,167664,148.793067,247.967292,12.310059,12.278831,0.926025,68.364980,114.178819,144.103576,268.524892
4,fold_2,baseline_persistence_seasonal,144710,207.209270,424.901161,18.241424,16.045526,0.776036,98.056594,142.853289,188.383758,399.541137
5,fold_2,baseline_moving_avg_24h,167664,431.173927,684.752969,44.866707,32.948452,0.435887,420.920574,327.104689,291.138808,685.531637
6,fold_3,baseline_persistence_simple,167664,103.062662,176.369193,11.762143,11.629671,0.926871,43.246325,79.504473,101.487699,188.012149
7,fold_3,baseline_persistence_seasonal,144710,206.039134,399.376306,25.574429,20.880688,0.523178,103.905371,155.621253,183.249639,381.378250
8,fold_3,baseline_moving_avg_24h,167664,287.619614,489.107422,40.678753,30.049826,0.437589,256.856538,259.393617,214.613406,419.614895
9,final_test,baseline_persistence_simple,142714,81.295094,141.178019,12.525153,12.312282,0.901959,35.922431,63.758093,80.835907,144.663443


In [9]:
# 20 cellules pilotes (dÃ©terministes) â€” utilisÃ©es par Prophet pour limiter le temps de calcul
pilot_cells = sorted(df["square_id"].unique().to_list())[:20]
print("Pilot cells count:", len(pilot_cells))
print("Pilot cells sample:", pilot_cells[:5])


Pilot cells count: 20
Pilot cells sample: [3756, 3757, 3856, 3857, 3953]


## ModÃ¨le principal : LightGBM (ou XGBoost fallback)

Je garde les features construites en phase 3 et j'applique exactement le mÃªme protocole walk-forward.


In [10]:
# Feature set for boosting
# We exclude identifiers/target to avoid duplicate selects and leakage-prone IDs.
exclude_cols = {"target_1h", "square_id", "slot_30m", "day_idx"}
feature_cols = [c for c in df.columns if c not in exclude_cols]

print("Number of features:", len(feature_cols))
print(feature_cols[:12], "...")


Number of features: 28
['internet_volume', 'sms_in', 'call_in', 'hour_slot', 'dow', 'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'is_weekend', 'lag_1', 'lag_2'] ...


In [11]:
def train_predict_boosting(X_train: np.ndarray, y_train: np.ndarray, X_eval: np.ndarray) -> np.ndarray:
    if HAVE_LGBM:
        model = lgb.LGBMRegressor(
            objective="mae",
            n_estimators=120,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        return model.predict(X_eval)

    if HAVE_XGB:
        model = xgb.XGBRegressor(
            objective="reg:absoluteerror",
            n_estimators=80,
            learning_rate=0.06,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            max_bin=64,
            random_state=42,
            n_jobs=2,
            tree_method="hist",
        )
        model.fit(X_train, y_train)
        return model.predict(X_eval)

    raise RuntimeError("Neither LightGBM nor XGBoost is available in this environment.")


In [12]:
boost_metrics_rows = []
boost_preds_frames = []

# Use all cells for fair comparison (same family of scope as requested)
boost_cells = sorted(df['square_id'].unique().to_list())
print('Boosting cells:', len(boost_cells))

for f in folds + [final_split]:
    print(f"Training boosting for {f['fold']}...", flush=True)
    train_df = filter_day_range(df, f["train_days"][0], f["train_days"][1]).filter(pl.col('square_id').is_in(boost_cells))
    eval_df = filter_day_range(df, f["eval_days"][0], f["eval_days"][1]).filter(pl.col('square_id').is_in(boost_cells))

    train_np = train_df.select(feature_cols + ["target_1h"]).to_numpy()
    eval_np = eval_df.select(feature_cols + ["target_1h"]).to_numpy()

    X_train = train_np[:, :-1].astype(np.float32)
    y_train = train_np[:, -1].astype(np.float32)
    X_eval = eval_np[:, :-1].astype(np.float32)
    y_true = eval_np[:, -1].astype(np.float32)

    y_pred = train_predict_boosting(X_train, y_train, X_eval)

    m = compute_metrics(y_true, y_pred)
    q = quantile_mae(y_true, y_pred)
    model_name = "lightgbm" if HAVE_LGBM else "xgboost"

    boost_metrics_rows.append({
        "split": f["fold"],
        "model": model_name,
        "n_samples": int(len(y_true)),
        **m,
        **q,
    })

    meta = eval_df.select(["square_id", "slot_30m", "day_idx", "target_1h"])
    pred_pl = meta.with_columns([
        pl.Series("pred_boost", y_pred),
        pl.lit(f["fold"]).alias("split"),
    ])
    boost_preds_frames.append(pred_pl)

    del train_np, eval_np, X_train, y_train, X_eval, y_true, y_pred, meta
    gc.collect()

boost_metrics = pd.DataFrame(boost_metrics_rows)
boost_predictions = pl.concat(boost_preds_frames)

boost_metrics


Boosting cells: 499
Training boosting for fold_1...


Training boosting for fold_2...
Training boosting for fold_3...
Training boosting for final_test...


,split,model,n_samples,MAE,RMSE,MAPE,sMAPE,R2,MAE_Q1,MAE_Q2,MAE_Q3,MAE_Q4
0,fold_1,xgboost,167664,141.613831,299.304976,12.092057,10.873067,0.909113,66.251266,86.593338,117.597649,296.013031
1,fold_2,xgboost,167664,116.401337,247.778592,12.110956,9.478053,0.926137,53.484486,75.626076,100.718506,235.776291
2,fold_3,xgboost,167664,99.479370,191.016524,14.445716,12.265470,0.914220,68.292427,86.014389,80.513924,163.096741
3,final_test,xgboost,142714,72.898720,138.872778,13.778128,11.640529,0.905134,41.770664,52.606251,65.256912,131.960251


## Skill score vs meilleure baseline

Je calcule le skill score par split :
`skill = 1 - MAE_model / MAE_best_baseline`.


In [ ]:
# Best baseline by split
best_baseline = (
    baseline_metrics
    .groupby("split", as_index=False)["MAE"]
    .min()
    .rename(columns={"MAE": "MAE_best_baseline"})
)

# Merge for boosting
boost_summary = boost_metrics.merge(best_baseline, on="split", how="left")
boost_summary["skill_vs_best_baseline"] = 1.0 - (boost_summary["MAE"] / boost_summary["MAE_best_baseline"])

# Merge for LSTM (peut Ãªtre exÃ©cutÃ© plus tard, donc garde-fou)
if 'lstm_metrics' in globals() and len(lstm_metrics) > 0:
    lstm_summary = lstm_metrics.merge(best_baseline, on='split', how='left')
    lstm_summary['skill_vs_best_baseline'] = 1.0 - (lstm_summary['MAE'] / lstm_summary['MAE_best_baseline'])
else:
    lstm_summary = pd.DataFrame(columns=['split', 'model', 'MAE', 'RMSE', 'n_samples', 'MAE_best_baseline', 'skill_vs_best_baseline'])
boost_summary


In [ ]:
# Consolidated metrics table
baseline_with_skill = baseline_metrics.merge(best_baseline, on="split", how="left")
baseline_with_skill["skill_vs_best_baseline"] = 1.0 - (baseline_with_skill["MAE"] / baseline_with_skill["MAE_best_baseline"])

all_metrics = pd.concat([
    baseline_with_skill,
    boost_summary,
    lstm_summary,
], ignore_index=True)

all_metrics = all_metrics.sort_values(["split", "MAE"]).reset_index(drop=True)
all_metrics


,split,model,n_samples,MAE,RMSE,MAE_best_baseline,skill_vs_best_baseline
0,final_test,xgboost,14300,52.789978,77.292006,81.295094,0.350638
1,final_test,baseline_persistence_simple,142714,81.295094,141.178019,81.295094,0.000000
2,final_test,baseline_persistence_seasonal,119760,140.039986,250.053992,81.295094,-0.722613
3,final_test,baseline_moving_avg_24h,142714,206.490933,350.393874,81.295094,-1.540017
4,fold_1,xgboost,16800,95.483589,139.233603,173.803258,0.450623
5,fold_1,baseline_persistence_simple,167664,173.803258,288.507110,173.803258,0.000000
6,fold_1,baseline_persistence_seasonal,144710,240.847636,472.725966,173.803258,-0.385749
7,fold_1,baseline_moving_avg_24h,167664,474.632060,746.867076,173.803258,-1.730858
8,fold_2,xgboost,16800,77.296104,105.863654,148.793067,0.480513
9,fold_2,baseline_persistence_simple,167664,148.793067,247.967292,148.793067,0.000000


## LSTM simple exÃ©cutable (PyTorch)

J'entraÃ®ne un LSTM sur des sÃ©quences de `internet_volume` brut (pas les lags), avec les mÃªmes fenÃªtres temporelles walk-forward.

Pour garder un temps de calcul raisonnable en local, l'entraÃ®nement reste sur un sous-ensemble de cellules lorsque c'est pertinent (voir paramÃ¨tre `lstm_cells` dans la cellule de code).


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

seq_len = 48
horizon = 2
lstm_cells = sorted(df['square_id'].unique().to_list())  # same scope as XGBoost

class TrafficLSTM(nn.Module):
    def __init__(self, input_size: int = 1, hidden: int = 64, layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def make_sequences(ts: np.ndarray, seq_len: int = 48, horizon: int = 2):
    X, y = [], []
    for i in range(len(ts) - seq_len - horizon + 1):
        X.append(ts[i:i + seq_len])
        y.append(ts[i + seq_len + horizon - 1])
    if len(X) == 0:
        return None, None
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def train_lstm_one_cell(train_series: np.ndarray, val_tail_size: int = 96):
    scaler = StandardScaler()
    train_norm = scaler.fit_transform(train_series.reshape(-1, 1)).ravel()

    split_idx = max(seq_len + horizon + 1, len(train_norm) - val_tail_size)
    tr = train_norm[:split_idx]
    va = train_norm[split_idx - seq_len - horizon + 1:]

    X_tr, y_tr = make_sequences(tr, seq_len, horizon)
    X_va, y_va = make_sequences(va, seq_len, horizon)
    if X_tr is None or X_va is None:
        return None, None

    X_tr_t = torch.tensor(X_tr).unsqueeze(-1)
    y_tr_t = torch.tensor(y_tr)
    X_va_t = torch.tensor(X_va).unsqueeze(-1)
    y_va_t = torch.tensor(y_va)

    model = TrafficLSTM()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=128, shuffle=True)

    best_val = float('inf')
    patience, patience_cnt = 10, 0
    best_state = None

    for _ in range(100):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va_t), y_va_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                break

    model.load_state_dict(best_state)
    return model, scaler


def predict_lstm_one_cell(model, scaler, eval_series: np.ndarray):
    eval_norm = scaler.transform(eval_series.reshape(-1, 1)).ravel()
    X_ev, y_ev = make_sequences(eval_norm, seq_len, horizon)
    if X_ev is None:
        return None, None
    with torch.no_grad():
        pred_norm = model(torch.tensor(X_ev).unsqueeze(-1)).numpy()
    y_true = scaler.inverse_transform(y_ev.reshape(-1, 1)).ravel()
    y_pred = scaler.inverse_transform(pred_norm.reshape(-1, 1)).ravel()
    return y_true, y_pred


lstm_metrics_rows = []
lstm_pred_frames = []

for f in folds + [final_split]:
    train_df = filter_day_range(df, f['train_days'][0], f['train_days'][1]).filter(pl.col('square_id').is_in(lstm_cells))
    eval_df = filter_day_range(df, f['eval_days'][0], f['eval_days'][1]).filter(pl.col('square_id').is_in(lstm_cells))

    y_all_true, y_all_pred = [], []
    pred_blocks = []

    for cell in lstm_cells:
        tr_cell = train_df.filter(pl.col('square_id') == cell).sort('slot_30m')
        ev_cell = eval_df.filter(pl.col('square_id') == cell).sort('slot_30m')
        ts_train = tr_cell['internet_volume'].to_numpy()
        ts_eval = ev_cell['internet_volume'].to_numpy()

        if len(ts_train) < 200 or len(ts_eval) < (seq_len + horizon):
            continue

        model, scaler = train_lstm_one_cell(ts_train)
        if model is None:
            continue

        y_true, y_pred = predict_lstm_one_cell(model, scaler, ts_eval)
        if y_true is None:
            continue

        y_all_true.append(y_true)
        y_all_pred.append(y_pred)

        ev_meta = ev_cell.select(['square_id', 'slot_30m', 'day_idx']).to_pandas().iloc[seq_len + horizon - 1:].copy()
        ev_meta['target_1h'] = y_true
        ev_meta['pred_lstm_corrected'] = y_pred
        ev_meta['split'] = f['fold']
        pred_blocks.append(pl.from_pandas(ev_meta))

    if len(y_all_true) == 0:
        continue

    y_true_concat = np.concatenate(y_all_true)
    y_pred_concat = np.concatenate(y_all_pred)
    m = compute_metrics(y_true_concat, y_pred_concat)
    q = quantile_mae(y_true_concat, y_pred_concat)

    lstm_metrics_rows.append({
        'split': f['fold'],
        'model': 'lstm_corrected_per_cell',
        'n_samples': int(len(y_true_concat)),
        **m,
        **q,
    })

    if pred_blocks:
        lstm_pred_frames.append(pl.concat(pred_blocks))

lstm_metrics = pd.DataFrame(lstm_metrics_rows)
lstm_pred_df = pl.concat(lstm_pred_frames) if lstm_pred_frames else pl.DataFrame()
print(lstm_metrics if len(lstm_metrics) else 'LSTM produced no rows')


In [9]:
# STL + XGBoost residual hybrid

def extrapolate_trend_seasonal(train_values: np.ndarray, horizon_len: int, period: int = 48):
    stl = STL(train_values, period=period, robust=True).fit()
    trend = stl.trend
    seasonal = stl.seasonal

    # trend extrapolation with local linear regression on last 3 days
    k = min(len(trend), period * 3)
    x = np.arange(k).reshape(-1, 1)
    y = trend[-k:]
    lr = LinearRegression().fit(x, y)
    x_future = np.arange(k, k + horizon_len).reshape(-1, 1)
    trend_future = lr.predict(x_future)

    # seasonal extrapolation by repeating last full season
    season_template = seasonal[-period:] if len(seasonal) >= period else np.resize(seasonal, period)
    reps = int(math.ceil(horizon_len / period))
    seasonal_future = np.tile(season_template, reps)[:horizon_len]

    return trend_future, seasonal_future


hybrid_rows = []
hybrid_preds = []

for f in folds + [final_split]:
    train_df = filter_day_range(df, f['train_days'][0], f['train_days'][1]).sort(['square_id', 'slot_30m'])
    eval_df = filter_day_range(df, f['eval_days'][0], f['eval_days'][1]).sort(['square_id', 'slot_30m'])

    eval_pred_blocks = []
    y_true_all, y_pred_all = [], []

    for cell in boost_cells:
        tr = train_df.filter(pl.col('square_id') == cell)
        ev = eval_df.filter(pl.col('square_id') == cell)
        if tr.height < 200 or ev.height == 0:
            continue

        tr_vals = tr['internet_volume'].to_numpy()
        ev_target = ev['target_1h'].to_numpy()

        tr_trend, tr_season = extrapolate_trend_seasonal(tr_vals, len(tr_vals), period=48)
        resid_train = tr_vals - tr_trend - tr_season

        ev_trend, ev_season = extrapolate_trend_seasonal(tr_vals, len(ev_target), period=48)

        X_train_r = tr.select(feature_cols).to_numpy().astype(np.float32)
        X_eval_r = ev.select(feature_cols).to_numpy().astype(np.float32)

        if HAVE_XGB:
            resid_model = xgb.XGBRegressor(
                objective='reg:absoluteerror',
                n_estimators=220,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                tree_method='hist',
                n_jobs=4,
            )
        else:
            resid_model = lgb.LGBMRegressor(objective='mae', n_estimators=220, learning_rate=0.05, random_state=42)

        resid_model.fit(X_train_r, resid_train)
        resid_pred = resid_model.predict(X_eval_r)
        final_pred = ev_trend + ev_season + resid_pred

        y_true_all.append(ev_target)
        y_pred_all.append(final_pred)

        block = ev.select(['square_id', 'slot_30m', 'day_idx', 'target_1h']).with_columns([
            pl.Series('pred_stl_xgb_residual', final_pred),
            pl.lit(f['fold']).alias('split'),
        ])
        eval_pred_blocks.append(block)

    if y_true_all:
        yt = np.concatenate(y_true_all)
        yp = np.concatenate(y_pred_all)
        m = compute_metrics(yt, yp)
        q = quantile_mae(yt, yp)
        hybrid_rows.append({'split': f['fold'], 'model': 'stl_xgboost_residual', 'n_samples': int(len(yt)), **m, **q})

    if eval_pred_blocks:
        hybrid_preds.append(pl.concat(eval_pred_blocks))

hybrid_metrics = pd.DataFrame(hybrid_rows)
hybrid_pred_df = pl.concat(hybrid_preds) if hybrid_preds else pl.DataFrame()
hybrid_metrics.head()

NameError: name 'boost_cells' is not defined

In [ ]:
# Prophet on pilot cells (daily + weekly seasonality)

prophet_cells = pilot_cells
prophet_rows = []
prophet_preds = []

if not HAVE_PROPHET:
    print('Prophet not available. Install with: %pip install prophet')
else:
    for f in folds + [final_split]:
        tr_df = filter_day_range(df, f['train_days'][0], f['train_days'][1]).filter(pl.col('square_id').is_in(prophet_cells))
        ev_df = filter_day_range(df, f['eval_days'][0], f['eval_days'][1]).filter(pl.col('square_id').is_in(prophet_cells))

        y_true_all, y_pred_all = [], []
        pred_blocks = []

        for cell in prophet_cells:
            tr = tr_df.filter(pl.col('square_id') == cell).sort('slot_30m').to_pandas()
            ev = ev_df.filter(pl.col('square_id') == cell).sort('slot_30m').to_pandas()
            if len(tr) < 100 or len(ev) == 0:
                continue

            tr_prophet = pd.DataFrame({
                'ds': pd.to_datetime(tr['slot_30m'], unit='s', utc=True).dt.tz_localize(None),
                'y': tr['internet_volume'].astype(float),
            })

            model = Prophet(
                daily_seasonality=True,
                weekly_seasonality=True,
                yearly_seasonality=False,
                seasonality_mode='multiplicative',
                changepoint_prior_scale=0.05,
            )
            model.fit(tr_prophet)

            future = model.make_future_dataframe(periods=len(ev), freq='30min', include_history=False)
            fcst = model.predict(future)
            pred = fcst['yhat'].to_numpy()

            y_true = ev['target_1h'].to_numpy(dtype=float)
            y_true_all.append(y_true)
            y_pred_all.append(pred)

            blk = pl.from_pandas(pd.DataFrame({
                'square_id': ev['square_id'].to_numpy(),
                'slot_30m': ev['slot_30m'].to_numpy(),
                'day_idx': ev['day_idx'].to_numpy(),
                'target_1h': y_true,
                'pred_prophet': pred,
                'split': f['fold'],
            }))
            pred_blocks.append(blk)

        if y_true_all:
            yt = np.concatenate(y_true_all)
            yp = np.concatenate(y_pred_all)
            m = compute_metrics(yt, yp)
            q = quantile_mae(yt, yp)
            prophet_rows.append({'split': f['fold'], 'model': 'prophet_daily_weekly', 'n_samples': int(len(yt)), **m, **q})

        if pred_blocks:
            prophet_preds.append(pl.concat(pred_blocks))

prophet_metrics = pd.DataFrame(prophet_rows)
prophet_pred_df = pl.concat(prophet_preds) if prophet_preds else pl.DataFrame()
prophet_metrics.head()

In [ ]:
# Recompute summaries with all models + unified skill score
best_baseline = (
    baseline_metrics
    .groupby('split', as_index=False)['MAE']
    .min()
    .rename(columns={'MAE': 'MAE_best_baseline'})
)


def aggregate_if_needed(dfm: pd.DataFrame, model_cols=('split', 'model')) -> pd.DataFrame:
    if dfm is None or len(dfm) == 0:
        return pd.DataFrame()
    cols = [c for c in ['MAE', 'RMSE', 'MAPE', 'sMAPE', 'R2', 'MAE_Q1', 'MAE_Q2', 'MAE_Q3', 'MAE_Q4', 'n_samples'] if c in dfm.columns]
    agg = {c: ('sum' if c == 'n_samples' else 'mean') for c in cols}
    return dfm.groupby(list(model_cols), as_index=False).agg(agg)


baseline_with_skill = baseline_metrics.merge(best_baseline, on='split', how='left')
baseline_with_skill['skill_vs_best_baseline'] = 1.0 - (baseline_with_skill['MAE'] / baseline_with_skill['MAE_best_baseline'])

boost_summary = aggregate_if_needed(boost_metrics).merge(best_baseline, on='split', how='left')
if len(boost_summary):
    boost_summary['skill_vs_best_baseline'] = 1.0 - (boost_summary['MAE'] / boost_summary['MAE_best_baseline'])

lstm_summary = aggregate_if_needed(lstm_metrics).merge(best_baseline, on='split', how='left') if len(lstm_metrics) else pd.DataFrame()
if len(lstm_summary):
    lstm_summary['skill_vs_best_baseline'] = 1.0 - (lstm_summary['MAE'] / lstm_summary['MAE_best_baseline'])

hybrid_summary = aggregate_if_needed(hybrid_metrics).merge(best_baseline, on='split', how='left') if len(hybrid_metrics) else pd.DataFrame()
if len(hybrid_summary):
    hybrid_summary['skill_vs_best_baseline'] = 1.0 - (hybrid_summary['MAE'] / hybrid_summary['MAE_best_baseline'])

prophet_summary = aggregate_if_needed(prophet_metrics).merge(best_baseline, on='split', how='left') if len(prophet_metrics) else pd.DataFrame()
if len(prophet_summary):
    prophet_summary['skill_vs_best_baseline'] = 1.0 - (prophet_summary['MAE'] / prophet_summary['MAE_best_baseline'])

all_metrics = pd.concat([
    baseline_with_skill,
    boost_summary,
    lstm_summary,
    hybrid_summary,
    prophet_summary,
], ignore_index=True)

all_metrics = all_metrics.sort_values(['split', 'MAE']).reset_index(drop=True)
all_metrics.head()

,split,model,n_samples,MAE,RMSE,MAE_best_baseline,skill_vs_best_baseline
0,final_test,xgboost,14300,52.789978,77.292006,81.295094,0.350638
1,final_test,baseline_persistence_simple,142714,81.295094,141.178019,81.295094,0.000000
2,final_test,baseline_persistence_seasonal,119760,140.039986,250.053992,81.295094,-0.722613
3,final_test,baseline_moving_avg_24h,142714,206.490933,350.393874,81.295094,-1.540017
4,final_test,lstm_pytorch_seq48,4760,491.624725,536.949369,81.295094,-5.047409


## Sauvegarde des sorties dans reports/

J'exporte les mÃ©triques et les prÃ©dictions pour traÃ§abilitÃ© et comparaison dans le rapport.


In [ ]:
# Save metrics
metrics_path = reports_dir / 'metrics_modelling_improved.csv'
lstm_metrics_path = reports_dir / 'metrics_lstm_corrected.csv'
hybrid_metrics_path = reports_dir / 'metrics_stl_xgb_hybrid.csv'
prophet_metrics_path = reports_dir / 'metrics_prophet.csv'

all_metrics.to_csv(metrics_path, index=False)
if len(lstm_metrics):
    lstm_metrics.to_csv(lstm_metrics_path, index=False)
if len(hybrid_metrics):
    hybrid_metrics.to_csv(hybrid_metrics_path, index=False)
if len(prophet_metrics):
    prophet_metrics.to_csv(prophet_metrics_path, index=False)

# Save predictions
baseline_pred_path = reports_dir / 'predictions_baselines.parquet'
boost_pred_path = reports_dir / 'predictions_boosting.parquet'
lstm_pred_path = reports_dir / 'predictions_lstm_corrected.parquet'
hybrid_pred_path = reports_dir / 'predictions_stl_xgb_hybrid.parquet'
prophet_pred_path = reports_dir / 'predictions_prophet.parquet'

baseline_predictions.write_parquet(baseline_pred_path)
boost_predictions.write_parquet(boost_pred_path)
if len(lstm_pred_df):
    lstm_pred_df.write_parquet(lstm_pred_path)
if len(hybrid_pred_df):
    hybrid_pred_df.write_parquet(hybrid_pred_path)
if len(prophet_pred_df):
    prophet_pred_df.write_parquet(prophet_pred_path)

print('Saved main metrics:', metrics_path)
print('Saved predictions in reports/ for baseline, boosting, LSTM, STL+XGB, Prophet')


## SynthÃ¨se lisible des rÃ©sultats

Ce tableau est le point d'entrÃ©e pour la phase de rÃ©daction :
- comparer les baselines,
- vÃ©rifier le gain du modÃ¨le principal via le skill score,
- situer les modÃ¨les additionnels (LSTM, STL+XGB, Prophet) par rapport aux baselines et au boosting.


In [ ]:
# Quick summary views
print('Best model per split (lower MAE is better):')
print(all_metrics.sort_values(['split', 'MAE']).groupby('split', as_index=False).first())

print('\nFull metrics table:')
print(all_metrics)


Best model per split (lower MAE is better):
        split    model  n_samples        MAE        RMSE  MAE_best_baseline  \
0  final_test  xgboost      14300  52.789978   77.292006          81.295094   
1      fold_1  xgboost      16800  95.483589  139.233603         173.803258   
2      fold_2  xgboost      16800  77.296104  105.863654         148.793067   
3      fold_3  xgboost      16800  92.848122  120.083695         103.062662   

   skill_vs_best_baseline  
0                0.350638  
1                0.450623  
2                0.480513  
3                0.099110  

Full metrics table:
         split                          model  n_samples         MAE  \
0   final_test                        xgboost      14300   52.789978   
1   final_test    baseline_persistence_simple     142714   81.295094   
2   final_test  baseline_persistence_seasonal     119760  140.039986   
3   final_test        baseline_moving_avg_24h     142714  206.490933   
4   final_test             lstm_pytorch